# Dense Structured Spectral Demo

This notebook demonstrates the real symmetric and complex Hermitian dense paths in `arb_mat` and `acb_mat`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "arbplusjax").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repo root")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import jax.numpy as jnp
from arbplusjax import arb_mat, acb_mat, acb_core, double_interval as di

In [ ]:
real_mid = jnp.array([[4.0, 1.0], [1.0, 3.0]], dtype=jnp.float64)
real_box = di.interval(real_mid, real_mid)
eigvals_i, eigvecs_i = arb_mat.arb_mat_eigh(real_box)
print('real symmetric eigvals midpoint:', di.midpoint(eigvals_i))
print('real symmetric eigvecs midpoint:\n', di.midpoint(eigvecs_i))

In [ ]:
complex_mid = jnp.array([[4.0 + 0.0j, 1.0 + 1.0j], [1.0 - 1.0j, 5.0 + 0.0j]], dtype=jnp.complex128)
complex_box = acb_core.acb_box(
    di.interval(jnp.real(complex_mid), jnp.real(complex_mid)),
    di.interval(jnp.imag(complex_mid), jnp.imag(complex_mid)),
)
eigvals_c, eigvecs_c = acb_mat.acb_mat_eigh(complex_box)
print('complex Hermitian eigvals midpoint:', acb_core.acb_midpoint(eigvals_c))
print('complex Hermitian eigvecs midpoint:\n', acb_core.acb_midpoint(eigvecs_c))

In [ ]:
rhs_real = di.interval(jnp.array([[1.0], [2.0]], dtype=jnp.float64), jnp.array([[1.0], [2.0]], dtype=jnp.float64))
rhs_complex = acb_core.acb_box(
    di.interval(jnp.array([[1.0], [2.0]], dtype=jnp.float64), jnp.array([[1.0], [2.0]], dtype=jnp.float64)),
    di.interval(jnp.array([[0.5], [-0.25]], dtype=jnp.float64), jnp.array([[0.5], [-0.25]], dtype=jnp.float64)),
)
print('SPD solve midpoint:', di.midpoint(arb_mat.arb_mat_spd_solve(real_box, rhs_real)))
print('HPD solve midpoint:', acb_core.acb_midpoint(acb_mat.acb_mat_hpd_solve(complex_box, rhs_complex)))